In [1]:
import pandas as pd
import numpy as np
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print('Train data shape:', train.shape)
print('Test data shape:', test.shape)
print(train.head())
print('\n--------------------\n')
print(train.info())

num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

print(f"数值型特征数量: {len(num_cols)}")
print(f"类别型特征数量: {len(cat_cols)}")

print("\n房价描述统计:")
print(train['SalePrice'].describe())

Train data shape: (1460, 81)
Test data shape: (1459, 80)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  S

C:\Users\m1553\AppData\Local\Temp\ipykernel_31604\1595413810.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include=['object']).columns.tolist()


In [2]:
y = np.log1p(train['SalePrice'])
train_fatures = train.drop(['Id', 'SalePrice'], axis=1)
test_features = test.drop(['Id'], axis=1)

all_data = pd.concat([train_fatures, test_features], axis=0).reset_index(drop=True)
print(f"合并后的数据集形状: {all_data.shape}")


合并后的数据集形状: (2919, 79)


In [3]:
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2']
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

zero_cols = ['GarageArea', 'GarageCars', 'TotalBsmtSF', 
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'MasVnrArea']
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

num_cols = all_data.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = all_data.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols:
    if all_data[col].isnull().sum() > 0:
        all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

for col in num_cols:
    if all_data[col].isnull().sum() > 0:
        all_data[col] = all_data[col].fillna(all_data[col].median())

print("剩余缺失值总数:", all_data.isnull().sum().sum())

剩余缺失值总数: 0


C:\Users\m1553\AppData\Local\Temp\ipykernel_31604\151050262.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = all_data.select_dtypes(include=['object', 'category']).columns.tolist()


In [4]:
all_data_encoded = pd.get_dummies(all_data, drop_first=True)
print(f"编码后特征数量: {all_data_encoded.shape[1]}")

编码后特征数量: 258


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

X = all_data_encoded.iloc[:len(train)]
X_test = all_data_encoded.iloc[len(train):]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
model.fit(X_train, y_train)

val_preds_log = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_preds_log))
print(f"本地验证集 RMSE 误差 (越低越好): {rmse:.4f}")

test_preds_log = model.predict(X_test)

test_preds = np.expm1(test_preds_log)

submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': test_preds
})
submission.to_csv('submission.csv', index=False)
print("基线文件已导出至 submission.csv")

本地验证集 RMSE 误差 (越低越好): 0.1465
基线文件已导出至 submission.csv
